# Beyond Audio — multi-seed neural-network stability audit

**Independent follow-up to the original Stage 2 experiment.** This notebook asks whether the conclusions from **CNN and MM early-fusion feedforward neural networks** depend strongly on the random seed.

- **Five prespecified seeds:** `42, 43, 44, 45, 46` — ten trained models in total (CNN and MM × five seeds).
- **Fixed original configuration:** same artist-disjoint partitions, same training-only scaling, same networks, AdamW parameters, batch size, max epochs, and early stopping as the verified Stage 2 notebook.
- **Model selection:** each run's checkpoint is selected **only** by validation macro-F1. No test-set tuning, no picking the best seed using test accuracy.
- **Reporting:** every seed, mean ± sample standard deviation, min–max, and descriptive comparisons with the fixed logistic-regression baselines. This is a **small seed-sensitivity study, not a significance test**.
- **Research limitation:** this is a follow-up designed after the original test results were inspected. Therefore it examines initialization stability *on the existing split*, not independent generalization or an untouched confirmatory experiment.

Requires the same Google Drive `MSD-I_project` files used for Stage 2. No external feature training is necessary, but it does train the ten downstream neural networks. A GPU runtime is helpful; CPU is supported. This notebook writes only to a **new output folder**.

## 1. Connect the original project and record the software environment
Run in Google Colab. If you have the project files under another root, set `MSDI_PROJECT_DIR` in the environment first.

In [ ]:
from pathlib import Path
import os, sys, platform, random, copy, json
import numpy as np
import pandas as pd
import sklearn, torch, joblib
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = Path(os.environ.get('MSDI_PROJECT_DIR', '/content/drive/MyDrive/MSD-I_project'))
CNN_DIR = PROJECT / 'finetuning_experiments' / 'multimodal_artist_disjoint'
MM_DIR  = PROJECT / 'finetuning_experiments' / 'oramas_representation_experiment'
STAGE2_DIR = PROJECT / 'finetuning_experiments' / 'reproducible_stage2_2026'
OUTPUT = PROJECT / 'finetuning_experiments' / 'multiseed_nn_stability_2026'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS = (42, 43, 44, 45, 46)  # fixed BEFORE looking at these runs' test results
MAX_EPOCHS, PATIENCE, BATCH_SIZE, MIN_DELTA = 100, 12, 128, 1e-4
CLASS_NAMES = ('Electronic', 'Jazz', 'Metal', 'Pop', 'Rap', 'Rock')
# Test baselines from the already-reported, validation-selected logistic regressions.
LR_TEST_REFERENCE = {'CNN': {'accuracy': 69.64, 'macro_f1': 68.68},
                     'MM':  {'accuracy': 70.20, 'macro_f1': 69.10}}

print('Project:', PROJECT)
print('New outputs:', OUTPUT)
print('Device:', DEVICE, ('(' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else ''))
print('Versions:', {'python': platform.python_version(), 'numpy': np.__version__,
                   'pandas': pd.__version__, 'sklearn': sklearn.__version__, 'torch': torch.__version__})
print('Prespecified seeds:', SEEDS)


## 2. Verify train/validation metadata and align CNN/MM feature arrays
No test arrays are loaded during the training and checkpoint-selection cells. The test data is read only in Section 5.

In [ ]:
SIZES = {'train': 8242, 'val': 1836, 'test': 1792}
REQUIRED_META = ('msd_track_id', 'msd_artist_id', 'genre', 'split')
labels, raw = {}, {}

def read_aligned_labels(split):
    p_cnn, p_mm = CNN_DIR / f'labels_{split}.csv', MM_DIR / f'labels_{split}.csv'
    if not p_cnn.is_file() or not p_mm.is_file():
        raise FileNotFoundError(f'Missing labels for {split}: {p_cnn} or {p_mm}')
    a = pd.read_csv(p_cnn, dtype={'msd_track_id': str, 'msd_artist_id': str})
    b = pd.read_csv(p_mm, dtype={'msd_track_id': str, 'msd_artist_id': str})
    for df in (a, b):
        assert set(REQUIRED_META) <= set(df.columns)
        assert len(df) == SIZES[split]
        assert df['msd_track_id'].is_unique
        assert df[list(REQUIRED_META)].notna().all().all()
        assert df['split'].eq(split).all()
    for col in REQUIRED_META:
        assert a[col].astype(str).equals(b[col].astype(str)), (split, col)
    return a.reset_index(drop=True)

def load_array(path, expected_shape):
    if not path.is_file():
        raise FileNotFoundError(path)
    x = np.load(path, allow_pickle=False)
    assert x.shape == expected_shape, (str(path), x.shape, expected_shape)
    assert np.isfinite(x).all(), f'Non-finite entries: {path}'
    return x

for split in ('train', 'val'):
    labels[split] = read_aligned_labels(split)
    n = SIZES[split]
    ca = load_array(CNN_DIR / f'X_{split}_audio.npy', (n, 2048))
    cv = load_array(CNN_DIR / f'X_{split}_visual.npy', (n, 2048))
    ma = load_array(MM_DIR / f'X_{split}_mm-audio.npy', (n, 200))
    mv = load_array(MM_DIR / f'X_{split}_mm-visual.npy', (n, 200))
    # Match the original Stage 2 pipeline: concatenate, cast float32, THEN scale.
    raw['CNN', split] = np.concatenate((ca, cv), axis=1).astype(np.float32)
    raw['MM', split] = np.concatenate((ma, mv), axis=1).astype(np.float32)

assert set(labels['train']['msd_artist_id']).isdisjoint(set(labels['val']['msd_artist_id']))
assert set(labels['train']['msd_track_id']).isdisjoint(set(labels['val']['msd_track_id']))
encoder = LabelEncoder()
y = {'train': encoder.fit_transform(labels['train']['genre'].astype(str)).astype(np.int64),
     'val': encoder.transform(labels['val']['genre'].astype(str)).astype(np.int64)}
assert tuple(encoder.classes_) == CLASS_NAMES
print('Aligned data verified; training and validation are artist-disjoint.')
print('Train:', len(y['train']), '| Val:', len(y['val']))
for model_name in ('CNN','MM'):
    print(model_name, raw[model_name,'train'].shape, raw[model_name,'val'].shape)


## 3. Original Stage 2 architecture and training-only preprocessing
The scaler is fitted once per representation on the training partition, then reused for *all seeds*. This is not a hyperparameter and does not use validation/test statistics.

In [ ]:
prepared, scalers = {}, {}
for name in ('CNN', 'MM'):
    scaler = StandardScaler()
    tr = scaler.fit_transform(raw[name, 'train']).astype(np.float32)
    va = scaler.transform(raw[name, 'val']).astype(np.float32)
    assert np.isfinite(tr).all() and np.isfinite(va).all()
    prepared[name] = {'train': tr, 'val': va}
    scalers[name] = scaler

class FusionMLP(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, dropout1, dropout2, num_classes=6):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Dropout(dropout1),
            nn.Linear(hidden1, hidden2), nn.ReLU(), nn.Dropout(dropout2),
            nn.Linear(hidden2, num_classes))
    def forward(self, x):
        return self.network(x)

CONFIGS = {
    'CNN': dict(input_dim=4096, hidden1=256, hidden2=64,
                dropout1=0.30, dropout2=0.20, lr=0.0005, weight_decay=0.01),
    'MM':  dict(input_dim=400, hidden1=128, hidden2=32,
                dropout1=0.40, dropout2=0.30, lr=0.0003, weight_decay=0.05),
}

def build_model(name):
    cfg = CONFIGS[name]
    return FusionMLP(**{k: v for k,v in cfg.items() if k not in ('lr','weight_decay')})

for name in CONFIGS:
    model = build_model(name)
    print(name, 'trainable params:', f'{sum(p.numel() for p in model.parameters()):,}')
    assert sum(p.numel() for p in model.parameters()) == (1065670 if name=='CNN' else 55654)

del model


## 4. Train all prespecified seeds using validation-only checkpoint selection
This loop intentionally matches the original Stage 2 training choices. Each run has its own checkpoint and history; the seed-42 model is included as a reference. **The test set is not loaded here.**

In [ ]:
def reset_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_loaders(name):
    ds_tr = TensorDataset(torch.from_numpy(prepared[name]['train']),
                          torch.from_numpy(y['train']))
    ds_va = TensorDataset(torch.from_numpy(prepared[name]['val']),
                          torch.from_numpy(y['val']))
    return (ds_tr, ds_va,
            DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=0),
            DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0))


def predict_codes(model, matrix, batch_size=256):
    model.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(matrix), batch_size):
            xb = torch.from_numpy(matrix[start:start+batch_size]).to(DEVICE)
            out.append(model(xb).argmax(dim=1).cpu().numpy())
    return np.concatenate(out)


def fit_one(name, seed):
    cfg = CONFIGS[name]
    ds_tr, ds_va, tr_loader, va_loader = make_loaders(name)
    # The original resets the global seed after loader creation, before model init.
    reset_seed(seed)
    model = build_model(name).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    best_f1, best_epoch, best_state = -float('inf'), 0, None
    stale, history = 0, []
    for epoch in range(1, MAX_EPOCHS+1):
        model.train()
        tr_loss_total, tr_pred, tr_true = 0.0, [], []
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            tr_loss_total += float(loss.item()) * len(yb)
            tr_pred.extend(logits.argmax(dim=1).detach().cpu().numpy())
            tr_true.extend(yb.cpu().numpy())
        tr_f1 = f1_score(tr_true, tr_pred, average='macro', zero_division=0)

        model.eval()
        va_loss_total, va_pred, va_true = 0.0, [], []
        with torch.no_grad():
            for xb, yb in va_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                loss = loss_fn(logits, yb)
                va_loss_total += float(loss.item()) * len(yb)
                va_pred.extend(logits.argmax(dim=1).cpu().numpy())
                va_true.extend(yb.cpu().numpy())
        va_f1 = f1_score(va_true, va_pred, average='macro', zero_division=0)
        history.append(dict(model=name, seed=seed, epoch=epoch,
                            train_loss=tr_loss_total/len(ds_tr), val_loss=va_loss_total/len(ds_va),
                            train_macro_f1=tr_f1, val_macro_f1=va_f1))
        if va_f1 > best_f1 + MIN_DELTA:
            best_f1, best_epoch = va_f1, epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if epoch==1 or epoch%5==0 or stale>=PATIENCE:
            print(f'  {name} seed {seed} epoch {epoch:2d}: val F1={va_f1:.4f}, best={best_f1:.4f} (epoch {best_epoch})')
        if stale >= PATIENCE:
            break
    assert best_state is not None
    model.load_state_dict(best_state)
    model.eval()
    restored_pred = predict_codes(model, prepared[name]['val'])
    restored_f1 = f1_score(y['val'], restored_pred, average='macro', zero_division=0)
    assert abs(restored_f1 - best_f1) < 1e-12

    ckpt = OUTPUT / f'{name.lower()}_seed{seed}_best.pt'
    torch.save(dict(model_state_dict=best_state,
                    model_name=name, seed=int(seed), best_epoch=best_epoch,
                    val_macro_f1=best_f1,
                    **{k: cfg[k] for k in ('input_dim','hidden1','hidden2','dropout1','dropout2')}), ckpt)
    pd.DataFrame(history).to_csv(OUTPUT / f'{name.lower()}_seed{seed}_history.csv', index=False)
    return dict(model=name, seed=seed, best_epoch=best_epoch,
                val_macro_f1_pct=100*best_f1,
                val_accuracy_pct=100*accuracy_score(y['val'], restored_pred),
                val_balanced_accuracy_pct=100*balanced_accuracy_score(y['val'], restored_pred),
                checkpoint_file=ckpt.name)

OUTPUT.mkdir(parents=True, exist_ok=True)
for name in ('CNN','MM'):
    joblib.dump(scalers[name], OUTPUT / f'{name.lower()}_train_scaler.joblib')

manifest = dict(project='Beyond Audio', purpose='follow-up multi-seed stability, not historical original',
                seeds=list(SEEDS), configurations=CONFIGS, max_epochs=MAX_EPOCHS,
                patience=PATIENCE, min_delta=MIN_DELTA, batch_size=BATCH_SIZE,
                python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                sklearn=sklearn.__version__, torch=torch.__version__, cuda=str(torch.version.cuda),
                gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None))
(OUTPUT / 'experiment_manifest.json').write_text(json.dumps(manifest, indent=2))

train_rows=[]
for name in ('CNN','MM'):
    for seed in SEEDS:
        print(f'\nTRAINING {name} / seed {seed}')
        train_rows.append(fit_one(name, seed))
validation_results = pd.DataFrame(train_rows)
validation_results.to_csv(OUTPUT / 'validation_results_all_seeds.csv', index=False)
print('\nALL SEEDS TRAINED. Validation-selected checkpoints:')
print(validation_results[['model','seed','best_epoch','val_macro_f1_pct','val_accuracy_pct']].to_string(index=False,float_format=lambda x:f'{x:.2f}'))
print('Saved to:', OUTPUT)


## 5. Final held-out test evaluation of **every** validation-selected checkpoint
**Do not change seeds, hyperparameters, or architecture based on these results.** Evaluating all prespecified seeds is a descriptive sensitivity analysis, not a procedure for selecting the best test result.

This cell deliberately loads the test set *after* Section 4 has saved every model checkpoint.

In [ ]:
assert len(validation_results)==2*len(SEEDS)
assert set(validation_results['model'])=={'CNN','MM'}
for name in ('CNN','MM'):
    assert set(validation_results.loc[validation_results.model==name, 'seed'])==set(SEEDS)

labels['test'] = read_aligned_labels('test')
for split in ('train','val'):
    assert set(labels['test']['msd_track_id']).isdisjoint(set(labels[split]['msd_track_id']))
    assert set(labels['test']['msd_artist_id']).isdisjoint(set(labels[split]['msd_artist_id']))
y['test'] = encoder.transform(labels['test']['genre'].astype(str)).astype(np.int64)

def test_matrix(name):
    n=SIZES['test']
    if name=='CNN':
        a=load_array(CNN_DIR/'X_test_audio.npy',(n,2048))
        v=load_array(CNN_DIR/'X_test_visual.npy',(n,2048))
    else:
        a=load_array(MM_DIR/'X_test_mm-audio.npy',(n,200))
        v=load_array(MM_DIR/'X_test_mm-visual.npy',(n,200))
    return np.concatenate((a,v),axis=1).astype(np.float32)

x_test={name:scalers[name].transform(test_matrix(name)).astype(np.float32) for name in ('CNN','MM')}
rows=[]
pred_cols={'msd_track_id':labels['test']['msd_track_id'],
           'msd_artist_id':labels['test']['msd_artist_id'],
           'genre':labels['test']['genre']}
for rec in validation_results.itertuples(index=False):
    name, seed=rec.model, int(rec.seed)
    ckpt=torch.load(OUTPUT / rec.checkpoint_file, map_location=DEVICE, weights_only=True)
    assert ckpt['model_name']==name and int(ckpt['seed'])==seed
    model=build_model(name).to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    pred=predict_codes(model,x_test[name])
    decoded=encoder.inverse_transform(pred)
    pred_cols[f'{name.lower()}_seed{seed}_prediction']=decoded
    rows.append(dict(model=name, seed=seed, best_epoch=rec.best_epoch,
                     val_macro_f1_pct=rec.val_macro_f1_pct,
                     val_accuracy_pct=rec.val_accuracy_pct,
                     test_accuracy_pct=100*accuracy_score(y['test'],pred),
                     test_balanced_accuracy_pct=100*balanced_accuracy_score(y['test'],pred),
                     test_macro_f1_pct=100*f1_score(y['test'],pred,average='macro',zero_division=0)))
    del model

test_results=pd.DataFrame(rows)
assert len(test_results)==2*len(SEEDS)
test_results.to_csv(OUTPUT/'test_metrics_all_seeds.csv',index=False)
pd.DataFrame(pred_cols).to_csv(OUTPUT/'test_predictions_all_seeds.csv',index=False)
print('All prespecified seeds, without test-based selection:')
print(test_results.to_string(index=False,float_format=lambda x:f'{x:.2f}'))
print('Saved predictions and metrics to:',OUTPUT)


## 6. Summarize seed stability and compare descriptively with logistic regression
The mean, sample standard deviation (`ddof=1`), minimum, and maximum are descriptive **across five chosen seeds**, not confidence intervals and not evidence of dataset generalization. Keep each seed in the full report even if one underperforms.

In [ ]:
from IPython.display import display
metrics=('val_macro_f1_pct','test_accuracy_pct','test_balanced_accuracy_pct','test_macro_f1_pct')
summary=[]
for name, group in test_results.groupby('model', sort=False):
    row={'model':name,'number_seeds':len(group)}
    for metric in metrics:
        v=group[metric].to_numpy(dtype=float)
        row[metric+'_mean']=v.mean()
        row[metric+'_sd']=v.std(ddof=1)
        row[metric+'_min']=v.min()
        row[metric+'_max']=v.max()
    row['best_epoch_min']=int(group.best_epoch.min())
    row['best_epoch_max']=int(group.best_epoch.max())
    row['runs_achieving_or_exceeding_lr_accuracy']=int(
        (group.test_accuracy_pct>=LR_TEST_REFERENCE[name]['accuracy']).sum())
    summary.append(row)
summary=pd.DataFrame(summary)
summary.to_csv(OUTPUT/'seed_stability_summary.csv',index=False)
print('Cross-seed summaries (percent):')
for rec in summary.itertuples(index=False):
    print(f'{rec.model}: val macro-F1={rec.val_macro_f1_pct_mean:.2f} ± {rec.val_macro_f1_pct_sd:.2f}; '
          f'test acc={rec.test_accuracy_pct_mean:.2f} ± {rec.test_accuracy_pct_sd:.2f}; '
          f'balanced acc={rec.test_balanced_accuracy_pct_mean:.2f} ± {rec.test_balanced_accuracy_pct_sd:.2f}; '
          f'test macro-F1={rec.test_macro_f1_pct_mean:.2f} ± {rec.test_macro_f1_pct_sd:.2f}')
    print(f'  Test acc min–max: {rec.test_accuracy_pct_min:.2f}–{rec.test_accuracy_pct_max:.2f}; '
          f'best epoch range: {rec.best_epoch_min}–{rec.best_epoch_max}; '
          f'accuracy ≥ fixed LR baseline in {rec.runs_achieving_or_exceeding_lr_accuracy}/{rec.number_seeds} runs')
print('\nFull numeric summary:')
display(summary.round(3))

# Diagnostic only: a seed-42 match is expected in a matching environment but is not guaranteed across hardware.
prior=STAGE2_DIR/'stage2_test_predictions.csv'
if prior.is_file():
    orig=pd.read_csv(prior,dtype={'msd_track_id':str})
    assert orig.msd_track_id.is_unique
    new=pd.DataFrame(pred_cols)
    for name in ('CNN','MM'):
        col=f'{name.lower()}_nn_prediction'
        if col in orig.columns:
            both=new[['msd_track_id',f'{name.lower()}_seed42_prediction']].merge(
                orig[['msd_track_id',col]],on='msd_track_id',validate='one_to_one',how='inner')
            assert len(both)==1792
            differences=(both[f'{name.lower()}_seed42_prediction']!=both[col]).sum()
            print(f'{name} seed 42 different test predictions vs Stage 2:',int(differences))
else:
    print('Stage 2 predictions not found: seed-42 historical comparison skipped.')


## 7. Optional publication-ready descriptive plot
This is a plot of **five runs**, not uncertainty intervals. Each dot is one test result; dashed lines indicate the fixed logistic-regression test accuracy already reported in the paper. Do **not** select a seed after looking at the plot.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,4.5))
for j,name in enumerate(('CNN','MM')):
    s=test_results[test_results.model==name].sort_values('seed')
    x=np.arange(len(s))+j*7
    ax.plot(x,s.test_accuracy_pct.to_numpy(),marker='o',label=name+' NN')
    ax.hlines(LR_TEST_REFERENCE[name]['accuracy'], x.min(), x.max(),
              linestyles='dashed',label=name+' logistic baseline')
ax.set_xticks([*range(5),*range(7,12)],
              [*map(str,SEEDS),*map(str,SEEDS)])
ax.set_xlabel('Prespecified random seed (CNN left, MM right)')
ax.set_ylabel('Test accuracy (%)')
ax.set_title('Test accuracy across random seeds (descriptive stability check)')
ax.grid(axis='y',alpha=.25)
ax.legend(fontsize=8,frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT/'seed_stability_test_accuracy.png',dpi=180)
plt.show()
plt.close(fig)


## 8. Reporting checklist

Use this exact distinction when writing the paper:

1. **What this experiment measures:** variation in performance caused by five initializations/shuffle sequences under a fixed data split and fixed hyperparameters.
2. **How models were selected:** independently for each seed using the *validation* macro-F1, with the original 12-epoch early stopping criterion.
3. **What to report:** all ten runs; mean, sample SD and range of test metrics; optional count above the previously selected LR baseline, descriptive only.
4. **What not to claim:** that these are confidence intervals, independent test replications, or proof of generalization to new artists/datasets. This extension was designed after observing previous results.
5. **How to interpret discrepancies:** differences in seed 42 against original predictions should be investigated as possible software/device nondeterminism, not silently replaced or optimized away.
6. **Reproducibility:** new files live in `finetuning_experiments/multiseed_nn_stability_2026`. Preserve previous outputs and report both the code and environment details.

**Before editing the paper:** share `test_metrics_all_seeds.csv` and `seed_stability_summary.csv`, or simply upload this executed notebook so we can inspect the complete outputs. Do not copy anticipated results into the paper before running the experiment.